In [ ]:
import requests
import pandas as pd
import os
# import keyboard

In [ ]:
# import time

# STEAMTOKEN = "B6FFF012DDFDF717841B22863BE764EA"
# TOKEN = "01e07e3b482524039d335543a96769e8bd5b718d"
# # Fetch full Steam app list and (optionally) try to map to IsThereAnyDeal IDs for a sample.
# SEARCH_URL = "https://api.isthereanydeal.com/games/search/v1"

# STEAM_APPLIST_URL = "https://api.steampowered.com/IStoreService/GetAppList/v1/"
# steam_params = {"key": STEAMTOKEN,
#                 "last_appid": 0,
#                 "max_results": 50000}

In [ ]:
# for i in range(5):  # Loop to handle pagination
#     r = requests.get(STEAM_APPLIST_URL, params = steam_params, timeout=30)
#     apps = r.json().get("response", {}).get("apps", [])

#     steam_df = pd.DataFrame(apps)[["appid", "name"]].drop_duplicates(subset="appid").sort_values("name").reset_index(drop=True)
#     steam_df.to_csv("steam_app_list.csv", mode='a', header=False, index=False)
#     steam_params["last_appid"] = int(steam_df["appid"].iloc[-1])
#     print("Steam apps saved:", len(steam_df))
#     data = pd.read_csv(r"D:\NEU\Time-series Analysis\steam_app_list.csv")
#     print(len(data))

#     # Remove duplicates from the accumulated CSV after each loop
#     all_apps = pd.read_csv("steam_app_list.csv", names=["appid", "name"], header=None)
#     all_apps = all_apps.drop_duplicates(subset=["appid"]).reset_index(drop=True)
#     all_apps.to_csv("steam_app_list.csv", index=False, header=False)

Steam apps saved: 1051
165785
Steam apps saved: 146
164880
Steam apps saved: 4
164738


KeyError: "None of [Index(['appid', 'name'], dtype='object')] are in the [columns]"

In [72]:
TOKEN = "01e07e3b482524039d335543a96769e8bd5b718d"

STOP_KEY = "="  # Set your desired stop key combination here
SEARCH_URL = "https://api.isthereanydeal.com/games/search/v1"
INFO_URL = "https://api.isthereanydeal.com/games/info/v2"
HISTORY_URL = "https://api.isthereanydeal.com/games/history/v2"
PRICES_URL = "https://api.isthereanydeal.com/games/prices/v3"

def _extract_list(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        data = payload.get("data")
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return [data]
        if isinstance(payload.get("results"), list):
            return payload["results"]
        if isinstance(payload.get("history"), list):
            return payload["history"]
    return []

def _first_item(payload):
    items = _extract_list(payload)
    return items[0] if items else None

def _to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _safe_get_amount(obj):
    if isinstance(obj, dict):
        return _to_float(obj.get("amount"))
    return _to_float(obj)

def _request_json(url, headers, params, timeout=30):
    r = requests.get(url, headers=headers, params=params, timeout=timeout)
    if r.status_code != 200:
        return None, r.status_code, r.text
    return r.json(), 200, ""


In [107]:
def build_isthereanydeal_datasets(game_names, token, country="US", since="2011-01-01T00:00:00Z", nondeals=1, gf_path="gfcheckpoint.csv", history_path="historycheckpoint.csv"):
    headers = {"Authorization": f"Bearer {token}"}
    game_info_rows = []
    price_history_rows = []

    existing_keys = set()
    if os.path.exists(gf_path):
        try:
            existing_df = pd.read_csv(gf_path, usecols=["query_name"])
            existing_keys = set(existing_df["query_name"].astype(str))
        except Exception:
            existing_keys = set()
        print(existing_keys)
        
    total_games = len(game_names)
    for idx, query_name in enumerate(game_names, start=1):
        print(f"[{idx}/{total_games}] Processing: {query_name}")
        if query_name in existing_keys:
            print(f"[{idx}/{total_games}] Skipped (already in CSV): {query_name}")
            continue
        # 1) Search and take the first result
        search_params = {"key": token, "title": query_name, "country": country}
        search_payload, status_code, err = _request_json(SEARCH_URL, headers, search_params)
        if status_code != 200:
            info_row = {
                "query_name": query_name,
                "game_title": query_name,
                "status": f"search_error_{status_code}",
                "error": err
            }
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{**info_row}]).to_csv(gf_path, mode="a", header=gw, index=False)
            game_info_rows.append({
                "game_title": query_name,
                "status": f"search_error_{status_code}",
                "error": err
            })
            print(f"[{idx}/{total_games}] Failed at search: {query_name} (status={status_code})")
            continue

        first_result = _first_item(search_payload)
        if not first_result:
            info_row = {
                "query_name": query_name,
                "game_title": query_name,
                "status": "no_search_result",
                "error": "No game matched this search term"
            }
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{**info_row}]).to_csv(gf_path, mode="a", header=gw, index=False)
            game_info_rows.append({
                "game_title": query_name,
                "status": "no_search_result",
                "error": "No game matched this search term"
            })
            print(f"[{idx}/{total_games}] No search result: {query_name}")
            continue

        game_id = str(first_result.get("id") or "")
        if not game_id:
            info_row = {
                "query_name": query_name,
                "game_title": query_name,
                "status": "missing_game_id",
                "error": "First search result did not include an id"
            }
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{**info_row}]).to_csv(gf_path, mode="a", header=gw, index=False)
            game_info_rows.append({
                "game_title": query_name,
                "status": "missing_game_id",
                "error": "First search result did not include an id"
            })
            print(f"[{idx}/{total_games}] Missing game id: {query_name}")
            continue
        
        # 2) Fetch game info
        info_params = {"key": token, "id": game_id, "country": country}
        info_payload, info_status, info_err = _request_json(INFO_URL, headers, info_params)
        if info_status != 200:
            info_row = {
                "query_name": query_name,
                "game_title": query_name,
                "status": f"info_error_{info_status}",
                "error": info_err
            }
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{**info_row}]).to_csv(gf_path, mode="a", header=gw, index=False)
            game_info_rows.append({
                "game_title": query_name,
                "status": f"info_error_{info_status}",
                "error": info_err
            })
            print(f"[{idx}/{total_games}] Failed at info endpoint: {query_name} (status={info_status})")
            continue

        game = _first_item(info_payload) or (info_payload if isinstance(info_payload, dict) else {})
        if isinstance(game, dict) and "data" in game and isinstance(game["data"], dict):
            game = game["data"]

        app_id = game.get("appid")
        title = game.get("title") or first_result.get("title") or query_name
        game_key = str(app_id) if app_id is not None else title
        tags = game.get("tags", []) or []
        tags_text = ", ".join(tags) if isinstance(tags, list) else str(tags)
        if "hentai" in tags_text:
            continue
        # 3) Fetch price history (long range)
        history_params = {
            "key": token,
            "id": game_id,
            "country": country,
            "since": since,
            "nondeals": nondeals
        }
        history_payload, history_status, history_err = _request_json(HISTORY_URL, headers, history_params)
        history_items = _extract_list(history_payload) if history_status == 200 else []

        # Optional: call /prices/v3 for latest snapshot, useful for current context
        prices_params = {"key": token, "id": game_id, "country": country}
        prices_payload, prices_status, _ = _request_json(PRICES_URL, headers, prices_params)
        latest_prices_count = len(_extract_list(prices_payload)) if prices_status == 200 else 0

        for entry in history_items:
            timestamp = entry.get("timestamp") or entry.get("time")
            parsed_ts = pd.to_datetime(timestamp, utc=True, errors="coerce")
            date_val = parsed_ts.tz_convert(None).normalize() if pd.notna(parsed_ts) else pd.NaT

            deal = entry.get("deal", {}) if isinstance(entry.get("deal", {}), dict) else {}
            price_obj = deal.get("price", {}) if isinstance(deal.get("price", {}), dict) else {}
            regular_obj = deal.get("regular", {})
            shop_obj = entry.get("shop", {}) if isinstance(entry.get("shop", {}), dict) else {}

            price = _safe_get_amount(price_obj)
            if price is None:
                price = _safe_get_amount(entry.get("amount"))

            regular_price = _safe_get_amount(regular_obj)
            if regular_price is None:
                regular_price = _safe_get_amount(entry.get("regular"))

            sales_percentage = _to_float(deal.get("cut", entry.get("cut")))
            if sales_percentage is None and regular_price and price is not None and regular_price > 0:
                sales_percentage = round((regular_price - price) / regular_price * 100, 2)

            shop_id = shop_obj.get("id", entry.get("shop_id"))
            shop_name = shop_obj.get("name", entry.get("shop_name", ""))

            price_row = {
                "game_key": game_key,
                "app_id": app_id,
                "game_title": title,
                "date": date_val,
                "price": price,
                "regular_price": regular_price,
                "sales_percentage": sales_percentage,
                "shop_id": shop_id,
                "shop_name": shop_name,
                "currency": price_obj.get("currency", entry.get("currency"))
            }
            price_history_rows.append(price_row)
            # append price row to history checkpoint CSV
            try:
                hw = not os.path.exists(history_path)
                pd.DataFrame([{
                    **price_row,
                    "date": pd.to_datetime(price_row["date"], errors="coerce").strftime("%Y-%m-%d")
                }]).to_csv(history_path, mode="a", header=hw, index=False)
            except Exception as e:
                print(f"[{idx}/{total_games}] Failed to append to history checkpoint: {e}")

        info_row = {
            "query_name": query_name,
            "game_key": game_key,
            "app_id": app_id,
            "game_title": title,
            "mature": game.get("mature"),
            "early_access": game.get("earlyAccess"),
            "release_date": game.get("releaseDate"),
            "tags": tags_text,
            "history_rows": len(history_items),
            "latest_prices_rows": latest_prices_count,
            "status": "ok" if history_status == 200 else f"history_error_{history_status}",
            "error": "" if history_status == 200 else history_err
        }
        game_info_rows.append(info_row)
        # append game info row to gf checkpoint CSV
        try:
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{
                **info_row,
                "release_date": pd.to_datetime(info_row["release_date"], errors="coerce").strftime("%Y-%m-%d")
            }]).to_csv(gf_path, mode="a", header=gw, index=False)
        except Exception as e:
            gw = not os.path.exists(gf_path)
            pd.DataFrame([{
                **info_row
            }]).to_csv(gf_path, mode="a", header=gw, index=False)
            print(f"[{idx}/{total_games}] Failed to append to gf checkpoint: {e}")
        
        print(f"[{idx}/{total_games}] Done: {title} | history_rows={len(history_items)}")

    game_info_df = pd.DataFrame(game_info_rows)
    price_history_df = pd.DataFrame(price_history_rows)

    if not price_history_df.empty:
        # Build a stable grouping key so duration is computed within each game+shop timeline
        price_history_df["shop_group"] = price_history_df["shop_id"].fillna(price_history_df["shop_name"]).fillna("unknown_shop")
        price_history_df = price_history_df.sort_values(["game_key", "shop_group", "date"]).reset_index(drop=True)

        # Sales duration = difference from current record (t) to next record (t+1)
        next_date = price_history_df.groupby(["game_key", "shop_group"])["date"].shift(-1)
        delta_days = (next_date - price_history_df["date"]).dt.total_seconds().div(86400)
        price_history_df["sales_duration_days"] = delta_days.round(2)
        # No discount means no sale duration
        price_history_df.loc[price_history_df["sales_percentage"].fillna(0).eq(0), "sales_duration_days"] = 0
        price_history_df = price_history_df.drop(columns=["shop_group"])

        # Historical low flag (without storing duplicated low value per row)
        min_price_by_game = price_history_df.groupby("game_key")["price"].transform("min")
        price_history_df["is_historical_low"] = price_history_df["price"].eq(min_price_by_game)

        # Merge price summary into the game info dataset
        summary_df = (
            price_history_df.groupby("game_key", as_index=False)
            .agg(
                first_price_record=("date", "min"),
                latest_price_record=("date", "max"),
                historical_low=("price", "min")
            )
        )
        game_info_df = game_info_df.merge(summary_df, on="game_key", how="left")

    # Final ordering requested: game title then date, both ascending
    if not price_history_df.empty:
        price_history_df = price_history_df.sort_values(["game_title", "date"], ascending=[True, True]).reset_index(drop=True)
        price_history_df["date"] = pd.to_datetime(price_history_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    if not game_info_df.empty:
        game_info_df = game_info_df.sort_values(["game_title"], ascending=[True]).reset_index(drop=True)
        game_info_df["first_price_record"] = pd.to_datetime(game_info_df["first_price_record"], errors="coerce").dt.strftime("%Y-%m-%d")
        game_info_df["latest_price_record"] = pd.to_datetime(game_info_df["latest_price_record"], errors="coerce").dt.strftime("%Y-%m-%d")

    # Remove internal keys from final exported datasets
    if "game_key" in game_info_df.columns:
        game_info_df = game_info_df.drop(columns=["game_key"])
    if "game_key" in price_history_df.columns:
        price_history_df = price_history_df.drop(columns=["game_key"])

    return game_info_df, price_history_df

In [90]:
import os

# extract game_name_list from steam_app_list.csv (fallback to existing all_apps DF if file not found)

if os.path.exists("steam_app_list.csv"):
    try:
        steam_apps = pd.read_csv("steam_app_list.csv", names=["appid", "name"], header=None)
        # if the file has a header row, re-read properly
        if steam_apps.iloc[0].astype(str).str.lower().str.contains("appid").any():
            steam_apps = pd.read_csv("steam_app_list.csv")
        steam_apps = steam_apps[~steam_apps["name"].astype(str).str.contains("!", na=False)].reset_index(drop=True)
    except Exception:
        steam_apps = pd.read_csv("steam_app_list.csv")
else:
    steam_apps = all_apps.copy()
    # drop rows whose name contains "!"
    
steam_apps.columns = ["appid", "name"]
game_name_list = (
    steam_apps["name"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(f"list size: {len(game_name_list)}")

list size: 159710


In [108]:
game_info_df, price_history_df = build_isthereanydeal_datasets(
    game_names=game_name_list,
    token = TOKEN)
print(price_history_df.shape)

{'"217" A Psychological Survival Thriller', '"BUTTS: The VR Experience"'}
[1/159710] Processing: "217" A Psychological Survival Thriller
[1/159710] Skipped (already in CSV): "217" A Psychological Survival Thriller
[2/159710] Processing: "BUTTS: The VR Experience"
[2/159710] Skipped (already in CSV): "BUTTS: The VR Experience"
[3/159710] Processing: "Buy The Game, I Have a Gun" -Sheesh-Man
[3/159710] No search result: "Buy The Game, I Have a Gun" -Sheesh-Man
[4/159710] Processing: "Glow Ball" - The billiard puzzle game
[4/159710] Done: "Glow Ball" - The billiard puzzle game | history_rows=251
[5/159710] Processing: "Going Up?"
[5/159710] Done: "Going Up?" | history_rows=3
[6/159710] Processing: "I Know This Place..?"  chapter I (prologue)
[6/159710] Done: I Know This Place..? (chapter II) | history_rows=19
[7/159710] Processing: "LIFE" not found;
[7/159710] Done: "LIFE" not found; | history_rows=1
[8/159710] Processing: "Otherworldly: Beginning of the Rift" Pre-Alpha
[8/159710] Done: "O

KeyboardInterrupt: 

In [ ]:
# Optional CSV export
game_info_df.to_csv("itad_game_info_dataset.csv", index=False)
price_history_df.to_csv("itad_price_history_dataset.csv", index=False)
print("Saved: itad_game_info_dataset.csv and itad_price_history_dataset.csv")

KeyboardInterrupt: 